<a href="https://colab.research.google.com/github/Arif118/Leukemia-Classification-from-Gene-Expression-/blob/main/Leukemia_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [ ]:
df = pd.read_csv('/content/leukemia_big.csv')
df.head(5)

,ALL,ALL.1,ALL.2,ALL.3,ALL.4,ALL.5,ALL.6,ALL.7,ALL.8,ALL.9,...,AML.15,AML.16,AML.17,AML.18,AML.19,AML.20,AML.21,AML.22,AML.23,AML.24
0,-1.533622,-0.867610,-0.433172,-1.671903,-1.187689,-1.127234,-1.045409,-0.106917,-1.198796,-1.190899,...,-0.436650,-1.274708,-0.681458,-0.876610,-0.624022,-0.431628,-1.435259,-0.671954,-1.013161,-0.969482
1,-1.235673,-1.275501,-1.184492,-1.596424,-1.335256,-1.113730,-0.800880,-0.745177,-0.849312,-1.190899,...,-0.915483,-1.354363,-0.653559,-1.096250,-1.066594,-1.335256,-1.204586,-0.751457,-0.889592,-1.080988
2,-0.333983,0.375927,-0.459196,-1.422571,-0.797493,-1.362768,-0.671954,-1.175674,0.320813,0.646610,...,-0.736156,-0.022153,-0.037455,-0.567335,-1.100749,-0.552938,-0.948874,-0.231657,-0.742163,-0.779500
3,0.488702,0.444011,0.436264,0.193353,0.235632,-0.360312,0.184941,0.425653,0.333983,0.235270,...,0.083781,0.356562,0.416241,0.533986,0.227505,0.416816,0.408202,0.326556,0.361813,0.298864
4,-1.300893,-1.229660,-1.325882,-1.818329,-1.311206,-1.513975,-1.651624,-1.339555,-0.593132,0.133302,...,-1.547444,-1.264475,-1.512318,-1.469583,-1.283472,-0.977672,-1.090178,-1.545120,-1.174272,-1.443183


In [ ]:
df.shape

(7128, 72)

In [ ]:
df = pd.read_csv('/content/leukemia_big.csv') # Reload df to ensure correct state
df = df.T
df.index = df.index.map(str) # Explicitly convert index elements to string
df["Target"] = df.index.map(lambda x: x.split('.')[0])
df.head()

,0,1,2,3,4,5,6,7,8,9,...,7119,7120,7121,7122,7123,7124,7125,7126,7127,Target
ALL,-1.533622,-1.235673,-0.333983,0.488702,-1.300893,-1.682668,-2.010995,-1.449186,0.035344,-1.088905,...,1.268788,-0.217954,0.255381,-1.057940,1.295992,0.733853,-0.301622,0.133657,-0.825596,ALL
ALL.1,-0.867610,-1.275501,0.375927,0.444011,-1.229660,-1.642072,0.572919,-1.588304,0.219574,0.119834,...,0.822880,-1.369024,0.542521,-0.796527,-0.218494,0.378380,-0.663166,-0.663166,-0.611045,ALL
ALL.2,-0.433172,-1.184492,-0.459196,0.436264,-1.325882,-1.407264,-0.264655,-1.147713,-0.573541,0.784512,...,0.642714,-0.466828,0.856140,-0.416816,1.132893,0.475669,-0.530138,1.566946,-0.805978,ALL
ALL.3,-1.671903,-1.596424,-1.422571,0.193353,-1.818329,-1.744469,-1.793197,-1.446178,-0.325815,-1.324191,...,0.462715,-0.585185,-0.181008,-0.611257,1.113077,0.148928,-0.625945,0.871972,-1.037246,ALL
ALL.4,-1.187689,-1.335256,-0.797493,0.235632,-1.311206,-1.654381,-1.441690,-1.058556,-0.582683,0.001758,...,0.750758,-0.380081,0.526298,-0.888026,0.719203,0.419502,-0.487514,0.358999,-0.742858,ALL


In [ ]:
#df.info()

In [ ]:
#df.isna().sum()

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
target = df.pop('Target')
df.insert(0, 'Target', target)
df = df.reset_index(drop=True)
df.head()

,Target,0,1,2,3,4,5,6,7,8,...,7118,7119,7120,7121,7122,7123,7124,7125,7126,7127
0,ALL,-1.533622,-1.235673,-0.333983,0.488702,-1.300893,-1.682668,-2.010995,-1.449186,0.035344,...,0.385567,1.268788,-0.217954,0.255381,-1.057940,1.295992,0.733853,-0.301622,0.133657,-0.825596
1,ALL,-0.867610,-1.275501,0.375927,0.444011,-1.229660,-1.642072,0.572919,-1.588304,0.219574,...,-0.158356,0.822880,-1.369024,0.542521,-0.796527,-0.218494,0.378380,-0.663166,-0.663166,-0.611045
2,ALL,-0.433172,-1.184492,-0.459196,0.436264,-1.325882,-1.407264,-0.264655,-1.147713,-0.573541,...,0.662728,0.642714,-0.466828,0.856140,-0.416816,1.132893,0.475669,-0.530138,1.566946,-0.805978
3,ALL,-1.671903,-1.596424,-1.422571,0.193353,-1.818329,-1.744469,-1.793197,-1.446178,-0.325815,...,0.161918,0.462715,-0.585185,-0.181008,-0.611257,1.113077,0.148928,-0.625945,0.871972,-1.037246
4,ALL,-1.187689,-1.335256,-0.797493,0.235632,-1.311206,-1.654381,-1.441690,-1.058556,-0.582683,...,0.176721,0.750758,-0.380081,0.526298,-0.888026,0.719203,0.419502,-0.487514,0.358999,-0.742858


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Target', axis=1), df['Target'], test_size=0.2, random_state=42)

In [ ]:
#pca = PCA()

#pca.fit(scaled_data)

#plt.plot(np.cumsum(pca.explained_variance_ratio_))
#.xlabel("Number of Components")
#plt.ylabel("Cumulative Explained Variance")
#plt.show()

In [ ]:
#pca=PCA(n_components=0.95)
#pca.fit(scaled_data)
#x_pca=pca.transform(scaled_data)

In [ ]:
#scaled_data.shape

(72, 7128)

In [ ]:
#x_pca.shape

(72, 70)

In [1]:
#x_pca = pd.DataFrame(x_pca, columns=['PC' + str(i) for i in range(1, 71)])
#x_pca['Target'] = target
#x_pca.head()

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('model', LogisticRegression())
])

In [ ]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=0.95)),
                ('model', LogisticRegression())])